# Livestock GHG emissions: per-animal aggregation

WRI's updated livestock GHG delivery (`GHG_Emissions_WRI/`) is disaggregated by animal type
(Buffalo, Cattle, Goat, Pig, Poultry, Sheep). The pipeline
(`pipelines/land_ghg_inventory/create_agriculture_zarr.py`) expects a single combined
livestock raster (`LIVESTOCK_COG_URI`), holding a per-hectare rate (kg CO2e/ha/yr) on a
~10km grid. This notebook sums the 6 animals into one such raster.

**Inputs used**: each animal's `Total_GHG_Emissions/..._ALL_kg_CO2e_ha_yr.tif` under
`GHG_Emissions_per_ha/`. Two things were verified beforehand (see the `GHG_Emissions_WRI`
exploration) and are not re-derived here:

- CH4 + N2O are already pre-summed into `Total_GHG_Emissions` by the data provider.
- Unlike Cornell's cropland rate (normalized by *physical cropland area* within a pixel,
  which overstated absolute totals ~5.3x when naively multiplied by full pixel area),
  livestock's per-ha rate is normalized by the **full geographic pixel area**
  (`absolute_total / per_ha_rate` reproduces the pixel's full area almost exactly, for both
  Cattle and Pig). So per-ha values are directly additive across animals here, with no
  area-weighting correction needed.

All 6 animals share an identical grid (4320x2160, 0.0833333 deg / ~10km, EPSG:4326, global
extent, float32, nodata=-9999), so no reprojection is needed to combine them.

Nodata is masked to NaN, and pixels are only NaN in the output where **every** animal is
NaN at that pixel (i.e. truly outside any livestock system's domain) -- a pixel with data in
some animals and not others still sums the animals that do have data.

In [1]:
import rasterio
import rioxarray as rio
import xarray as xr

# --- hardcoded, since this notebook doesn't have access to the pipelines package ---

REPO_ROOT = ".."  # run from notebooks/
PER_HA_DIR = f"{REPO_ROOT}/GHG_Emissions_WRI/GHG_Emissions_per_ha"

ANIMALS = ["Buffalo", "Cattle", "Goat", "Pig", "Poultry", "Sheep"]

ANIMAL_COG_PATHS = {
    animal: (
        f"{PER_HA_DIR}/{animal}/Total_GHG_Emissions/"
        f"Total_GHG_kg_CO2e_yr_{animal}_ALL_kg_CO2e_ha_yr.tif"
    )
    for animal in ANIMALS
}

# current pipeline input (see pipelines/land_ghg_inventory/create_agriculture_zarr.py),
# used below only as a magnitude sanity check against the newly aggregated raster.
LIVESTOCK_COG_URI = (
    "s3://gfw2-data/climate/AFOLU_flux_model/livestock_emissions/"
    "raw__from_Cornell/20260731_emis_per_ha_only/Total_GHG_Emissions/"
    "Tot_CO2eq_kg_livestock_GHG_emissions_kgCO2e_ha.tif"
)

OUTPUT_PATH = f"{REPO_ROOT}/GHG_Emissions_WRI/Total_GHG_kg_CO2e_ha_yr_AllAnimals.tif"

ANIMAL_COG_PATHS

{'Buffalo': '../GHG_Emissions_WRI/GHG_Emissions_per_ha/Buffalo/Total_GHG_Emissions/Total_GHG_kg_CO2e_yr_Buffalo_ALL_kg_CO2e_ha_yr.tif',
 'Cattle': '../GHG_Emissions_WRI/GHG_Emissions_per_ha/Cattle/Total_GHG_Emissions/Total_GHG_kg_CO2e_yr_Cattle_ALL_kg_CO2e_ha_yr.tif',
 'Goat': '../GHG_Emissions_WRI/GHG_Emissions_per_ha/Goat/Total_GHG_Emissions/Total_GHG_kg_CO2e_yr_Goat_ALL_kg_CO2e_ha_yr.tif',
 'Pig': '../GHG_Emissions_WRI/GHG_Emissions_per_ha/Pig/Total_GHG_Emissions/Total_GHG_kg_CO2e_yr_Pig_ALL_kg_CO2e_ha_yr.tif',
 'Poultry': '../GHG_Emissions_WRI/GHG_Emissions_per_ha/Poultry/Total_GHG_Emissions/Total_GHG_kg_CO2e_yr_Poultry_ALL_kg_CO2e_ha_yr.tif',
 'Sheep': '../GHG_Emissions_WRI/GHG_Emissions_per_ha/Sheep/Total_GHG_Emissions/Total_GHG_kg_CO2e_yr_Sheep_ALL_kg_CO2e_ha_yr.tif'}

## Load each animal's per-ha total and mask nodata to NaN

Files are small (4320x2160, ~10MB each) and local, so no dask chunking / distributed
cluster is needed here -- just read them directly into memory.

In [2]:
import numpy as np


def load_animal_per_ha(path: str) -> xr.DataArray:
    da = rio.open_rasterio(path)
    if "band" in da.dims:
        da = da.isel(band=0, drop=True)
    nodata = da.rio.nodata
    if nodata is not None:
        da = da.where(da != nodata)  # -> NaN
    return da


animal_arrays = {animal: load_animal_per_ha(p) for animal, p in ANIMAL_COG_PATHS.items()}

# sanity check: all 6 animals must be on the identical grid before summing. Transform
# equality is checked with a tolerance since Poultry's affine differs from the others by
# float precision noise only (e.g. bottom=-89.99999999999994 vs -90.0), not a real
# misalignment.
reference = next(iter(animal_arrays.values()))
for animal, da in animal_arrays.items():
    assert da.shape == reference.shape, f"{animal} shape mismatch: {da.shape} vs {reference.shape}"
    assert da.rio.crs == reference.rio.crs, f"{animal} CRS mismatch"
    assert np.allclose(
        tuple(da.rio.transform())[:6], tuple(reference.rio.transform())[:6], atol=1e-6
    ), f"{animal} transform mismatch"

print("grid:", reference.shape, reference.rio.crs, reference.rio.transform())

grid: (2160, 4320) EPSG:4326 | 0.08, 0.00,-180.00|
| 0.00,-0.08, 90.00|
| 0.00, 0.00, 1.00|


## Sum across animals

`skipna=True` treats a NaN at one animal's pixel as "no emissions from that animal," not as
missing data for the whole pixel -- so a pixel with data in some animals and not others still
sums the animals that do have data. A pixel is only NaN in the output where **every** animal
is NaN there.

In [3]:
# snap every animal onto the reference's exact x/y coordinates before stacking, since
# Poultry's coordinates differ from the others by float precision noise (see grid check
# above) and xr.concat aligns on coordinate values, not just shape.
aligned_arrays = [
    animal_arrays[a].assign_coords(x=reference.x, y=reference.y) for a in ANIMALS
]

stacked = xr.concat(
    aligned_arrays,
    dim=xr.DataArray(ANIMALS, dims="animal", name="animal"),
)

valid_count = stacked.notnull().sum(dim="animal")
total_per_ha = stacked.sum(dim="animal", skipna=True).where(valid_count > 0)

total_per_ha = total_per_ha.rio.write_crs(reference.rio.crs)
total_per_ha.rio.write_transform(reference.rio.transform(), inplace=True)
total_per_ha.rio.write_nodata(float("nan"), inplace=True)

total_per_ha

<xarray.DataArray (y: 2160, x: 4320)> Size: 37MB
array([[nan, nan, nan, ..., nan, nan, nan],
       [nan, nan, nan, ..., nan, nan, nan],
       [nan, nan, nan, ..., nan, nan, nan],
       ...,
       [nan, nan, nan, ..., nan, nan, nan],
       [nan, nan, nan, ..., nan, nan, nan],
       [nan, nan, nan, ..., nan, nan, nan]],
      shape=(2160, 4320), dtype=float32)
Coordinates:
  * x            (x) float64 35kB -180.0 -179.9 -179.8 ... 179.8 179.9 180.0
  * y            (y) float64 17kB 89.96 89.88 89.79 ... -89.79 -89.87 -89.96
    spatial_ref  int64 8B 0
Attributes:
    _FillValue:  nan

## QC: sums per animal vs. aggregated total, and vs. the live pipeline input

The aggregated global sum should be close to the sum of the 6 individual animal sums
(small differences can arise only from `skipna` at partial-coverage pixels, i.e. pixels
where some but not all animals report data). We also compare magnitude against the COG
currently wired into `create_agriculture_zarr.py` as `LIVESTOCK_COG_URI`, to see how the
new WRI total compares to what's live in the pipeline today.

In [4]:
animal_sums = {animal: float(da.sum(skipna=True)) for animal, da in animal_arrays.items()}
sum_of_animal_sums = sum(animal_sums.values())
aggregated_sum = float(total_per_ha.sum(skipna=True))

for animal, s in animal_sums.items():
    print(f"{animal:>10s}: {s:,.0f} kg CO2e/ha/yr (raster-wide sum)")

print()
print(f"sum of animal sums:  {sum_of_animal_sums:,.0f}")
print(f"aggregated total:    {aggregated_sum:,.0f}")
print(f"difference:          {(aggregated_sum - sum_of_animal_sums) / sum_of_animal_sums * 100:+.6f}%")

   Buffalo: 67,005,268 kg CO2e/ha/yr (raster-wide sum)
    Cattle: 372,489,760 kg CO2e/ha/yr (raster-wide sum)
      Goat: 31,871,694 kg CO2e/ha/yr (raster-wide sum)
       Pig: 41,917,208 kg CO2e/ha/yr (raster-wide sum)
   Poultry: 17,895,452 kg CO2e/ha/yr (raster-wide sum)
     Sheep: 36,273,240 kg CO2e/ha/yr (raster-wide sum)

sum of animal sums:  567,452,622
aggregated total:    567,451,712
difference:          -0.000160%


In [5]:
def sum_cog_kg_ha(cog_uri: str) -> float:
    with rasterio.Env(AWS_REQUEST_PAYER="requester"):
        src = rio.open_rasterio(cog_uri, chunks={"x": 10000, "y": 10000})
    if "band" in src.dims:
        src = src.isel(band=0, drop=True)
    nodata = src.rio.nodata
    if nodata is not None:
        src = src.where(src != nodata)
    return float(src.sum(skipna=True).compute())


try:
    live_pipeline_sum = sum_cog_kg_ha(LIVESTOCK_COG_URI)
    print(f"current LIVESTOCK_COG_URI sum: {live_pipeline_sum:,.0f} kg CO2e/ha/yr")
    print(f"new WRI aggregated sum:        {aggregated_sum:,.0f} kg CO2e/ha/yr")
    print(f"ratio (new / current):         {aggregated_sum / live_pipeline_sum:.4f}")
except Exception as exc:
    print(f"skipping S3 comparison against LIVESTOCK_COG_URI (no AWS access here): {exc}")

skipping S3 comparison against LIVESTOCK_COG_URI (no AWS access here): AWS_SECRET_ACCESS_KEY and AWS_NO_SIGN_REQUEST configuration options not defined, and /Users/solomon.negusse/.aws/credentials not filled


## Write the aggregated raster locally

Written with NaN nodata (not the source `-9999` sentinel) per the target format for this
deliverable. This is a local file for review -- promoting it into the pipeline (a new
versioned S3 COG replacing/extending `LIVESTOCK_COG_URI`, and any resulting change to
`create_agriculture_zarr.py`) is a separate follow-up once QC'd.

In [ ]:
total_per_ha = total_per_ha.astype("float32")
total_per_ha.rio.to_raster(OUTPUT_PATH, dtype="float32", nodata=float("nan"))

print(f"wrote {OUTPUT_PATH}") 

wrote ../GHG_Emissions_WRI/Total_GHG_kg_CO2e_ha_yr_AllAnimals.tif
